# Notebook 02 — Salary Data
## NBA Contract Value Index (CVI)

This notebook scrapes salary data from ESPN and Basketball-Reference,
cleans it, and merges it onto the BBRef stats master from notebook 01.
The output is our primary working dataset used in all downstream notebooks.

---

### What This Notebook Does

| Step | Action | Output |
|------|--------|--------|
| 1 | Load BBRef master from notebook 01 | `master` DataFrame |
| 2 | Define name fix functions (unicode + suffix mismatches) | `fix_player_names()` |
| 3 | Scrape current contracts from BBRef contracts page | `contracts_wide` |
| 4 | Calculate contract summary metrics | `SALARY_TOTAL_M`, `YEARS_REMAINING`, `SALARY_AVG_PER_YR_M` |
| 5 | Reshape contracts to long format | `contracts_long` |
| 6 | Scrape historical salaries from ESPN (all 5 seasons) | `espn_all` |
| 7 | Clean ESPN salary data | `espn_clean` |
| 8 | Merge ESPN salary onto master | 93.3% match rate |
| 9 | Fix remaining name mismatches | 93.3% → final match rate |
| 10 | Add BBRef contract summary metrics | `SALARY_TOTAL_M` etc. |
| 11 | Fix DATA_TYPE labels | training vs current |
| 12 | Save final dataset | `data/processed/master_with_salary.csv` |

---

### Data Sources
- **Basketball-Reference contracts page** — current + future year salaries
  - basketball-reference.com/contracts/players.html
- **ESPN salary pages** — historical annual salaries (2021-22 through 2025-26)
  - espn.com/nba/salaries/_/year/{year}/page/{page}

---

### Output
- `data/processed/master_with_salary.csv`
- One row per player-season
- 61 columns — all BBRef stats + salary metrics
- 2668 rows across 5 seasons
- 93.3% salary coverage (6.7% unmatched = fringe/two-way players)

---

### Salary Metrics Added

| Column | Description |
|--------|-------------|
| `SALARY_M` | Annual salary in millions for that season |
| `SALARY_TOTAL_M` | Total guaranteed money remaining on contract |
| `YEARS_REMAINING` | Number of contract years left including current |
| `SALARY_AVG_PER_YR_M` | Total contract value / years remaining |

---

### Key Decisions Made
- Used ESPN over HoopsHype — HoopsHype blocked scraping, ESPN returned clean data
- Left join on PLAYER_NAME + SEASON — keeps all players even without salary match
- Fixed names on BOTH sides of the join — ESPN and BBRef use different conventions
- Kept expiring contracts (YEARS_REMAINING = 1) as valid rows — low risk signal
- Fringe players (BPM < -1.5, unmatched) accepted as data gap — won't affect model

---

### Match Rate by Season

| Season | Matched | Total | Rate |
|--------|---------|-------|------|
| 2021-22 | 491 | 528 | 93% |
| 2022-23 | 503 | 528 | 95% |
| 2023-24 | 494 | 532 | 93% |
| 2024-25 | 510 | 543 | 94% |
| 2025-26 | 486 | 532 | 91% |

---

### Notes for V2
- Add contract type (player option, team option, two-way, qualifying offer)
- Expiring contracts should get a cap efficiency boost in feature engineering
- Consider Spotrac as backup salary source for better two-way contract coverage
- Anthony Davis NaN on SALARY_TOTAL_M = expiring deal, handle in feature engineering

In [33]:
import pandas as pd
import numpy as np
import os
import warnings
import requests
from bs4 import BeautifulSoup
import unicodedata
warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)

print("✓ Imports ready")

✓ Imports ready


## Load BBRef master

In [34]:
master = pd.read_csv("../data/raw/bbref_master.csv")
print(f"✓ Master loaded: {master.shape}")
print(f"Seasons: {sorted(master['SEASON'].unique())}")
print(f"Players: {master['PLAYER_NAME'].nunique()} unique")

✓ Master loaded: (2663, 55)
Seasons: ['2021-22', '2022-23', '2023-24', '2024-25', '2025-26']
Players: 758 unique


## Name fix functions (needed for matching)

In [35]:
def fix_player_names(name):
    """
    Convert accented unicode characters to ASCII equivalents.
    Handles encoding issues from BBRef HTML scraping.
    """
    if pd.isna(name):
        return name
    return unicodedata.normalize("NFKD", str(name)).encode("ascii", "ignore").decode("utf-8").strip()

# Manual fixes for names that unicode normalization can't handle
# These are the exact broken strings we found in notebook 01
name_fixes_exact = {
    "Nikola JokiA"       : "Nikola Jokic",
    "Luka DonAiA"        : "Luka Doncic",
    "Bogdan BogdanoviA"  : "Bogdan Bogdanovic",
    "Bojan BogdanoviA"   : "Bojan Bogdanovic",
    "Boban MarjanoviA"   : "Boban Marjanovic",
    "Goran DragiA"       : "Goran Dragic",
    "Jusuf NurkiA"       : "Jusuf Nurkic",
    "Nikola JoviA"       : "Nikola Jovic",
    "Nikola VuAeviA"     : "Nikola Vucevic",
    "Vasilije MiciA"     : "Vasilije Micic",
    "Karlo MatkoviA"     : "Karlo Matkovic",
    "Kristaps PorziAAis" : "Kristaps Porzingis",
    "Dario A ariA"       : "Dario Saric",
    "Luka A amaniA"      : "Luka Samanic",
    "DAvis BertAns"      : "Davis Bertans",
    "Egor DNmin"         : "Egor Demin",
    "TomAA SatoranskA12" : "Tomas Satoransky",
    "VAt KrejAA"         : "Vit Krejci",
}

print("✓ Name fix functions ready")

✓ Name fix functions ready


## Scrape BBref salary data

In [36]:
def scrape_bbref_contracts():
    """
    Scrape full contract data from Basketball-Reference contracts page.
    Captures every year of every player's contract — not just current season.
    This gives us total value, years remaining, and future salary commitments
    which are all critical for the CVI model's multi-year risk assessment.
    """
    url     = "https://www.basketball-reference.com/contracts/players.html"
    headers = {"User-Agent": "Mozilla/5.0"}
    
    resp = requests.get(url, headers=headers)
    time.sleep(3)
    
    # Parse the HTML table
    tables = pd.read_html(resp.text)
    df     = tables[0].copy()
    
    # Flatten multi-level column headers
    # BBRef uses tuples like ('Salary', '2025-26') and ('Unnamed', 'Player')
    new_cols = []
    for col in df.columns:
        if "Unnamed" in str(col[0]):
            # Identity columns — use just the second level
            new_cols.append(col[1].upper())
        else:
            # Salary columns — combine to 'SALARY_2025-26' format
            new_cols.append(f"SALARY_{col[1]}")
    df.columns = new_cols
    
    # Remove repeated header rows BBRef inserts mid-table
    df = df[df["PLAYER"] != "Player"].copy()
    df = df[df["RK"].notna()].copy()
    df = df.drop(columns=["RK"], errors="ignore")
    
    # Standardize column names
    df = df.rename(columns={
        "PLAYER" : "PLAYER_NAME",
        "TM"     : "TEAM",
    })
    
    # Fix broken player name encodings
    df["PLAYER_NAME"] = df["PLAYER_NAME"].apply(fix_player_names)
    df["PLAYER_NAME"] = df["PLAYER_NAME"].replace(name_fixes_exact)
    
    # Identify all salary year columns
    salary_cols = [c for c in df.columns if c.startswith("SALARY_")]
    print(f"  Contract years found: {salary_cols}")
    
    # Clean salary columns — strip $, commas, option flags (P/T/Q)
    for col in salary_cols:
        df[col] = (df[col]
                   .astype(str)
                   .str.replace(r"[^\d]", "", regex=True)  # digits only
                   .replace("", np.nan)
                   .astype(float))
    
    print(f"✓ Scraped {len(df)} players")
    return df, salary_cols

contracts_wide, salary_cols = scrape_bbref_contracts()
contracts_wide.head(10)

  Contract years found: ['SALARY_2025-26', 'SALARY_2026-27', 'SALARY_2027-28', 'SALARY_2028-29', 'SALARY_2029-30', 'SALARY_2030-31']
✓ Scraped 527 players


,PLAYER_NAME,TEAM,SALARY_2025-26,SALARY_2026-27,SALARY_2027-28,SALARY_2028-29,SALARY_2029-30,SALARY_2030-31,GUARANTEED
0,Stephen Curry,GSW,59606817.0,62587158.0,NaN,NaN,NaN,NaN,"$122,193,975"
1,Joel Embiid,PHI,55224526.0,58100000.0,62748000.0,67396000.0,NaN,NaN,"$176,072,526"
2,Nikola Jokic,DEN,55224526.0,59033114.0,62841702.0,NaN,NaN,NaN,"$114,257,640"
3,Kevin Durant,HOU,54708609.0,43902439.0,46097561.0,NaN,NaN,NaN,"$98,611,048"
4,Jayson Tatum,BOS,54126450.0,58456566.0,62786682.0,67116798.0,71446914.0,NaN,"$242,486,496"
5,Anthony Davis,WAS,54126450.0,58456566.0,62786682.0,NaN,NaN,NaN,"$112,583,016"
6,Giannis Antetokounmpo,MIL,54126450.0,58456566.0,62786682.0,NaN,NaN,NaN,"$112,583,016"
7,Jimmy Butler,GSW,54126450.0,56832773.0,NaN,NaN,NaN,NaN,"$110,959,223"
8,Jaylen Brown,BOS,53142264.0,57078728.0,61015192.0,64951656.0,NaN,NaN,"$236,187,840"
9,Devin Booker,PHO,53142264.0,57078728.0,61015192.0,64065952.0,69191228.0,NaN,"$235,302,136"


## Calculate contract summary metrics 

In [37]:
def calculate_contract_metrics(df, salary_cols):
    """
    From the wide contract table, calculate:
      SALARY_THIS_YEAR  — current season salary (2025-26)
      SALARY_TOTAL      — total guaranteed money remaining on contract
      SALARY_AVG_PER_YR — average annual value (total / years remaining)
      YEARS_REMAINING   — number of contract years left including current
    
    These metrics power several CVI features:
      - SALARY_THIS_YEAR → market comparison feature
      - SALARY_TOTAL     → CBA/apron impact feature  
      - YEARS_REMAINING  → age/trajectory risk multiplier
      - SALARY_AVG_PER_YR → the fairest single number for contract value
    """
    df = df.copy()
    
    # Current season salary
    current_col = "SALARY_2025-26"
    if current_col in df.columns:
        df["SALARY_THIS_YEAR"] = df[current_col]
    else:
        # Fallback to first salary column if 2025-26 not present
        df["SALARY_THIS_YEAR"] = df[salary_cols[0]]
    
    # Total guaranteed remaining — sum all non-null salary years
    df["SALARY_TOTAL"] = df[salary_cols].sum(axis=1, skipna=True)
    
    # Years remaining — count non-null, non-zero salary years
    df["YEARS_REMAINING"] = df[salary_cols].apply(
        lambda row: (row > 0).sum(), axis=1
    )
    
    # Average annual value
    df["SALARY_AVG_PER_YR"] = np.where(
        df["YEARS_REMAINING"] > 0,
        df["SALARY_TOTAL"] / df["YEARS_REMAINING"],
        np.nan
    )
    
    # Convert to millions for readability
    df["SALARY_THIS_YEAR_M"] = (df["SALARY_THIS_YEAR"] / 1_000_000).round(2)
    df["SALARY_TOTAL_M"]     = (df["SALARY_TOTAL"]     / 1_000_000).round(2)
    df["SALARY_AVG_PER_YR_M"]= (df["SALARY_AVG_PER_YR"]/ 1_000_000).round(2)
    
    print(f"✓ Contract metrics calculated")
    print(f"\nTop 10 by total contract value:")
    print(df[["PLAYER_NAME", "TEAM", "SALARY_THIS_YEAR_M", 
              "SALARY_TOTAL_M", "YEARS_REMAINING", "SALARY_AVG_PER_YR_M"]]
          .sort_values("SALARY_TOTAL_M", ascending=False)
          .head(10)
          .to_string(index=False))
    
    return df

contracts = calculate_contract_metrics(contracts_wide, salary_cols)

✓ Contract metrics calculated

Top 10 by total contract value:
             PLAYER_NAME TEAM  SALARY_THIS_YEAR_M  SALARY_TOTAL_M  YEARS_REMAINING  SALARY_AVG_PER_YR_M
 Shai Gilgeous-Alexander  OKC               38.33          352.44                6                58.74
            Jayson Tatum  BOS               54.13          313.93                5                62.79
            Devin Booker  PHO               53.14          304.49                5                60.90
             Evan Mobley  CLE               46.39          269.09                5                53.82
         Cade Cunningham  DET               46.39          269.09                5                53.82
            De'Aaron Fox  SAS               37.10          260.20                5                52.04
          Paolo Banchero  ORL               15.33          256.03                6                42.67
           Chet Holmgren  OKC               13.73          254.43                6                42.41
 

## Reshape to long format for historical merge

In [38]:
def reshape_contracts_long(df, salary_cols):
    """
    Convert wide contract table to long format.
    
    Wide format (one row per player):
      Player | SALARY_2025-26 | SALARY_2026-27 | SALARY_2027-28
    
    Long format (one row per player-season):
      Player | SEASON | SALARY | SALARY_M
    
    Long format is needed to join onto our BBRef stats master
    which is also in long format (one row per player-season).
    We keep the contract summary metrics on every row so they're
    always available regardless of which season we're looking at.
    """
    # Columns to carry through to every row
    id_cols = ["PLAYER_NAME", "TEAM", 
               "SALARY_THIS_YEAR_M", "SALARY_TOTAL_M", 
               "YEARS_REMAINING", "SALARY_AVG_PER_YR_M"]
    
    # Keep only id cols + salary year cols
    keep_cols = [c for c in id_cols if c in df.columns] + salary_cols
    df_slim   = df[keep_cols].copy()
    
    # Melt salary year columns into rows
    long = df_slim.melt(
        id_vars    = [c for c in id_cols if c in df.columns],
        value_vars = salary_cols,
        var_name   = "SEASON_RAW",
        value_name = "SALARY"
    )
    
    # Extract clean season string from column name
    # e.g. "SALARY_2025-26" → "2025-26"
    long["SEASON"] = long["SEASON_RAW"].str.replace("SALARY_", "", regex=False)
    long = long.drop(columns=["SEASON_RAW"])
    
    # Convert to millions
    long["SALARY_M"] = (long["SALARY"] / 1_000_000).round(2)
    
    # Drop rows where salary is null or zero (unfilled future years)
    long = long[long["SALARY"] > 0].dropna(subset=["SALARY"]).copy()
    long = long.sort_values(["PLAYER_NAME", "SEASON"]).reset_index(drop=True)
    
    print(f"✓ Long format: {len(long)} rows")
    print(f"  Seasons: {sorted(long['SEASON'].unique())}")
    return long

contracts_long = reshape_contracts_long(contracts, salary_cols)
contracts_long.head(10)

✓ Long format: 1242 rows
  Seasons: ['2025-26', '2026-27', '2027-28', '2028-29', '2029-30', '2030-31']


,PLAYER_NAME,TEAM,SALARY_THIS_YEAR_M,SALARY_TOTAL_M,YEARS_REMAINING,SALARY_AVG_PER_YR_M,SALARY,SEASON,SALARY_M
0,A.J. Green,MIL,2.30,47.30,5,9.46,2301587.0,2025-26,2.30
1,A.J. Green,MIL,2.30,47.30,5,9.46,10044644.0,2026-27,10.04
2,A.J. Green,MIL,2.30,47.30,5,9.46,10848215.0,2027-28,10.85
3,A.J. Green,MIL,2.30,47.30,5,9.46,11651786.0,2028-29,11.65
4,A.J. Green,MIL,2.30,47.30,5,9.46,12455355.0,2029-30,12.46
5,AJ Johnson,DAL,3.09,11.82,3,3.94,3090480.0,2025-26,3.09
6,AJ Johnson,DAL,3.09,11.82,3,3.94,3237120.0,2026-27,3.24
7,AJ Johnson,DAL,3.09,11.82,3,3.94,5493394.0,2027-28,5.49
8,Aaron Gordon,DEN,22.84,131.89,4,32.97,22841455.0,2025-26,22.84
9,Aaron Gordon,DEN,22.84,131.89,4,32.97,33658037.0,2026-27,33.66


## Merge onto Master

In [39]:
def merge_salary_onto_master(master_df, contracts_long_df, contracts_wide_df):
    """
    Join salary data onto BBRef stats master table.
    
    Two join strategies:
    1. Historical seasons (2021-22 to 2024-25) — join on PLAYER_NAME + SEASON
       from the long format contracts table
    2. Current season (2025-26) — same join, but also attach full contract
       summary metrics (total value, years remaining, avg per year)
    
    Left join keeps all players in master even if no salary found.
    We track match rate and surface unmatched players for manual review.
    """
    # Step 1 — join salary by player + season
    merged = master_df.merge(
        contracts_long_df[[
            "PLAYER_NAME", "SEASON", 
            "SALARY_M", "SALARY",
            "SALARY_THIS_YEAR_M", "SALARY_TOTAL_M",
            "YEARS_REMAINING", "SALARY_AVG_PER_YR_M"
        ]],
        on  = ["PLAYER_NAME", "SEASON"],
        how = "left"
    )
    
    # Match rate report
    matched   = merged["SALARY_M"].notna().sum()
    unmatched = merged["SALARY_M"].isna().sum()
    pct       = matched / len(merged) * 100
    
    print(f"✓ Total rows:       {len(merged)}")
    print(f"✓ Salary matched:   {matched} ({pct:.1f}%)")
    print(f"✗ No salary found:  {unmatched} ({100-pct:.1f}%)")
    
    return merged

master = merge_salary_onto_master(master, contracts_long, contracts)

✓ Total rows:       2688
✓ Salary matched:   526 (19.6%)
✗ No salary found:  2162 (80.4%)


In [40]:
# Surface unmatched players so we can fix name mismatches
# Focus on 2025-26 since that's our current scoring season
unmatched = master[
    (master["SALARY_M"].isna()) &
    (master["SEASON"] == "2025-26")
][["PLAYER_NAME", "TEAM", "BPM", "VORP"]].sort_values("BPM", ascending=False)

print(f"Unmatched in 2025-26: {len(unmatched)}")
print(f"\nTop unmatched by BPM (most important to fix):")
print(unmatched.head(20).to_string(index=False))

Unmatched in 2025-26: 31

Top unmatched by BPM (most important to fix):
      PLAYER_NAME TEAM  BPM  VORP
      Javon Small  MEM  1.3   0.7
   Ron Harper Jr.  BOS  0.5   0.2
  Branden Carlson  OKC  0.4   0.3
   Jamaree Bouyea  PHO  0.2   0.3
 Tristan Vukcevic  WAS -0.1   0.3
      A.J. Lawson  TOR -1.3   0.0
   Dylan Cardwell  SAC -1.8   0.0
       Jamal Cain  ORL -2.0   0.0
  Quenton Jackson  IND -2.1   0.0
 Chris Youngblood  2TM -2.2   0.0
       Pete Nance  MIL -2.3  -0.1
    Alijah Martin  TOR -2.5   0.0
 Brooks Barnhizer  OKC -2.5   0.0
    Isaiah Livers  PHO -2.6   0.0
   Oscar Tshiebwe  UTA -2.7  -0.1
     Moussa Cisse  DAL -2.7  -0.1
 Chris Youngblood  OKC -2.8   0.0
       Caleb Love  POR -3.1  -0.3
    Tyson Etienne  BRK -3.2  -0.1
    Jamir Watkins  WAS -3.3  -0.3


## Scrape historical salaries from BBRef team pages

In [41]:
# ESPN paginates salary data — each page has ~44 players
# URL format: /nba/salaries/_/seasontype/1/page/2 for page 2
# Let's scrape all pages and stack them

def scrape_espn_salaries(season_year=2026):
    """
    Scrape all salary pages from ESPN.
    
    ESPN paginates results ~44 players per page.
    We loop through pages until we get an empty table.
    
    Parameters:
        season_year: the year the season ends
                     e.g. 2026 = 2025-26 season
    """
    headers = {
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }
    
    all_frames = []
    page       = 1
    
    while True:
        # ESPN salary URL with page number and season year
        url = f"https://www.espn.com/nba/salaries/_/year/{season_year}/page/{page}"
        print(f"  Scraping page {page}...")
        
        try:
            resp = requests.get(url, headers=headers, timeout=10)
            time.sleep(2)  # be respectful
            
            # If no table found we've gone past the last page
            if "<table" not in resp.text:
                print(f"  No table on page {page} — done")
                break
                
            tables = pd.read_html(resp.text)
            if not tables:
                print(f"  No tables parsed on page {page} — done")
                break
            
            df = tables[0].copy()
            
            # ESPN uses numeric column indexes — rename them
            df.columns = ["RK", "NAME", "TEAM", "SALARY"]
            
            # Remove header rows that repeat mid-table
            # (ESPN repeats the header row every ~10 players)
            df = df[df["RK"] != "RK"].copy()
            df = df[df["NAME"] != "NAME"].copy()
            df = df[df["SALARY"] != "SALARY"].copy()
            
            # If table is empty after cleaning we're done
            if len(df) == 0:
                print(f"  Empty table on page {page} — done")
                break
            
            all_frames.append(df)
            print(f"  ✓ Page {page}: {len(df)} players")
            page += 1
            
        except Exception as e:
            print(f"  ✗ Error on page {page}: {e}")
            break
    
    if not all_frames:
        print("✗ No data scraped")
        return None
    
    # Stack all pages together
    result = pd.concat(all_frames, ignore_index=True)
    
    # Clean player name — ESPN adds position like "Stephen Curry, G"
    # Split on comma and take just the name part
    result["PLAYER_NAME"] = result["NAME"].str.split(",").str[0].str.strip()
    
    # Clean salary — remove $ and commas → float
    result["SALARY"] = (result["SALARY"]
                        .astype(str)
                        .str.replace(r"[^\d]", "", regex=True)
                        .replace("", np.nan)
                        .astype(float))
    result["SALARY_M"] = (result["SALARY"] / 1_000_000).round(2)
    
    # Fix encoding issues in player names
    result["PLAYER_NAME"] = result["PLAYER_NAME"].apply(fix_player_names)
    result["PLAYER_NAME"] = result["PLAYER_NAME"].replace(name_fixes_exact)
    
    # Drop rows with no salary
    result = result[result["SALARY"] > 0].dropna(subset=["SALARY"]).copy()
    result = result.reset_index(drop=True)
    
    print(f"\n✓ Total players scraped: {len(result)}")
    return result[["PLAYER_NAME", "TEAM", "SALARY", "SALARY_M"]]

# Test with current season first
print("Scraping 2025-26 salaries from ESPN...")
espn_2526 = scrape_espn_salaries(season_year=2026)
print(espn_2526.head(10))

Scraping 2025-26 salaries from ESPN...
  Scraping page 1...
  ✓ Page 1: 40 players
  Scraping page 2...
  ✓ Page 2: 40 players
  Scraping page 3...
  ✓ Page 3: 40 players
  Scraping page 4...
  ✓ Page 4: 40 players
  Scraping page 5...
  ✓ Page 5: 40 players
  Scraping page 6...
  ✓ Page 6: 40 players
  Scraping page 7...
  ✓ Page 7: 40 players
  Scraping page 8...
  ✓ Page 8: 40 players
  Scraping page 9...
  ✓ Page 9: 40 players
  Scraping page 10...
  ✓ Page 10: 40 players
  Scraping page 11...
  ✓ Page 11: 40 players
  Scraping page 12...
  ✓ Page 12: 35 players
  Scraping page 13...
  Empty table on page 13 — done

✓ Total players scraped: 475
             PLAYER_NAME                   TEAM      SALARY  SALARY_M
0          Stephen Curry  Golden State Warriors  59606817.0     59.61
1            Joel Embiid     Philadelphia 76ers  55224526.0     55.22
2           Nikola Jokic         Denver Nuggets  55224526.0     55.22
3           Kevin Durant        Houston Rockets  54708609.0    

475 players scraped perfectly — names are clean, salaries look right. Now let's pull all 5 seasons:

### Pull all 5 seasons

In [42]:
# Pull all seasons from ESPN
# ESPN year parameter = the year the season ENDS
# e.g. 2022 = 2021-22 season

espn_seasons = {
    "2021-22": 2022,
    "2022-23": 2023,
    "2023-24": 2024,
    "2024-25": 2025,
    "2025-26": 2026,
}

espn_frames = []

for season_label, year in espn_seasons.items():
    print(f"\n── {season_label} ──")
    df = scrape_espn_salaries(season_year=year)
    
    if df is not None:
        # Add season label to each row
        df["SEASON"] = season_label
        espn_frames.append(df)

# Stack all seasons together
espn_all = pd.concat(espn_frames, ignore_index=True)

print(f"\n✓ Total ESPN salary rows: {len(espn_all)}")
print(f"\nPlayers per season:")
print(espn_all.groupby("SEASON")["PLAYER_NAME"].count().to_string())


── 2021-22 ──
  Scraping page 1...
  ✓ Page 1: 40 players
  Scraping page 2...
  ✓ Page 2: 40 players
  Scraping page 3...
  ✓ Page 3: 40 players
  Scraping page 4...
  ✓ Page 4: 40 players
  Scraping page 5...
  ✓ Page 5: 40 players
  Scraping page 6...
  ✓ Page 6: 40 players
  Scraping page 7...
  ✓ Page 7: 40 players
  Scraping page 8...
  ✓ Page 8: 40 players
  Scraping page 9...
  ✓ Page 9: 40 players
  Scraping page 10...
  ✓ Page 10: 40 players
  Scraping page 11...
  ✓ Page 11: 40 players
  Scraping page 12...
  ✓ Page 12: 40 players
  Scraping page 13...
  ✓ Page 13: 17 players
  Scraping page 14...
  Empty table on page 14 — done

✓ Total players scraped: 497

── 2022-23 ──
  Scraping page 1...
  ✓ Page 1: 40 players
  Scraping page 2...
  ✓ Page 2: 40 players
  Scraping page 3...
  ✓ Page 3: 40 players
  Scraping page 4...
  ✓ Page 4: 40 players
  Scraping page 5...
  ✓ Page 5: 40 players
  Scraping page 6...
  ✓ Page 6: 40 players
  Scraping page 7...
  ✓ Page 7: 40 player

That's a great pull — 2467 rows across all 5 seasons. Takes about 3-4mins to run. Has about 12 pages of scraping.  Let's clean it properly before merging. ESPN Historical salary match looks great going forward. 

## Clean ESPN historical salary data

In [44]:
def clean_espn_salaries(df):
    """
    Clean the ESPN salary DataFrame before merging onto master.
    
    Issues to fix:
    1. Duplicate players — ESPN sometimes lists a player twice if they
       were on multiple teams. We keep the higher salary row (their
       primary contract, not a 10-day deal).
    2. Name mismatches — ESPN uses slightly different names than BBRef
       e.g. "Jimmy Butler III" vs "Jimmy Butler"
          "Jaren Jackson Jr." vs "Jaren Jackson"
          "Marcus Morris Sr." vs "Marcus Morris"
    3. Salary of 0 or NaN — two-way contract players sometimes show $0
    4. Team name format — ESPN uses full city names, BBRef uses 
       abbreviations. We'll keep ESPN team as a separate column
       since our master already has team abbreviations from BBRef.
    """
    cleaned = df.copy()
    
    # ── Step 1: Remove rows with no salary ──────────────────────────
    # These are usually two-way or exhibit players with no guaranteed money
    cleaned = cleaned[cleaned["SALARY"] > 0].copy()
    cleaned = cleaned.dropna(subset=["SALARY", "PLAYER_NAME"]).copy()
    
    # ── Step 2: Fix common name mismatches between ESPN and BBRef ────
    # ESPN often includes suffixes (Jr., Sr., III) that BBRef drops
    # We standardize to match BBRef's naming convention
    espn_name_fixes = {
        "Jimmy Butler III"      : "Jimmy Butler",
        "Jaren Jackson Jr."     : "Jaren Jackson",
        "Marcus Morris Sr."     : "Marcus Morris",
        "Larry Nance Jr."       : "Larry Nance",
        "Otto Porter Jr."       : "Otto Porter",
        "Gary Payton II"        : "Gary Payton",
        "Wendell Carter Jr."    : "Wendell Carter",
        "Kevin Porter Jr."      : "Kevin Porter",
        "Michael Porter Jr."    : "Michael Porter",
        "Kenyon Martin Jr."     : "Kenyon Martin",
        "Derrick Jones Jr."     : "Derrick Jones",
        "Dennis Schroder"       : "Dennis Schroder",
        "Luka Doncic"           : "Luka Doncic",
        "Nikola Jokic"          : "Nikola Jokic",
        "Mo Bamba"              : "Mohamed Bamba",
        "Moe Wagner"            : "Moritz Wagner",
        "Sviatoslav Mykhailiuk" : "Svi Mykhailiuk",
        "Jabari Smith"          : "Jabari Smith Jr.",
        "Xavier Tillman"        : "Xavier Tillman Sr.",
        "Kelly Oubre"           : "Kelly Oubre Jr.",
        "Nic Claxton"           : "Nicolas Claxton",
        "GG Jackson"            : "Gregory Jackson",
    }
    cleaned["PLAYER_NAME"] = cleaned["PLAYER_NAME"].replace(espn_name_fixes)
    
    # ── Step 3: Handle duplicate player-season rows ──────────────────
    # If a player appears twice in the same season keep the higher salary
    # This handles mid-season trades where ESPN lists both contracts
    before = len(cleaned)
    cleaned = (cleaned
               .sort_values("SALARY", ascending=False)
               .drop_duplicates(subset=["PLAYER_NAME", "SEASON"], keep="first")
               .copy())
    dupes_removed = before - len(cleaned)
    print(f"  Duplicates removed: {dupes_removed}")
    
    # ── Step 4: Strip whitespace from all string columns ─────────────
    # Prevents invisible space characters from breaking joins
    str_cols = cleaned.select_dtypes(include="object").columns
    for col in str_cols:
        cleaned[col] = cleaned[col].str.strip()
    
    # ── Step 5: Final sort and reset index ───────────────────────────
    cleaned = cleaned.sort_values(
        ["SEASON", "SALARY"], ascending=[True, False]
    ).reset_index(drop=True)
    
    print(f"  ✓ Cleaned shape: {cleaned.shape}")
    print(f"  ✓ Seasons: {sorted(cleaned['SEASON'].unique())}")
    print(f"  ✓ Salary range: ${cleaned['SALARY_M'].min()}M — ${cleaned['SALARY_M'].max()}M")
    return cleaned

print("Cleaning ESPN salary data...")
espn_clean = clean_espn_salaries(espn_all)

print(f"\nSample after cleaning:")
print(espn_clean[["PLAYER_NAME", "TEAM", "SEASON", "SALARY_M"]].head(10).to_string(index=False))

Cleaning ESPN salary data...
  Duplicates removed: 1
  ✓ Cleaned shape: (2466, 5)
  ✓ Seasons: ['2021-22', '2022-23', '2023-24', '2024-25', '2025-26']
  ✓ Salary range: $0.01M — $59.61M

Sample after cleaning:
           PLAYER_NAME                    TEAM   SEASON  SALARY_M
         Stephen Curry   Golden State Warriors  2021-22     45.78
          James Harden      Philadelphia 76ers  2021-22     44.31
             John Wall         Houston Rockets  2021-22     44.31
     Russell Westbrook      Los Angeles Lakers  2021-22     44.21
          Kevin Durant           Brooklyn Nets  2021-22     42.02
          LeBron James      Los Angeles Lakers  2021-22     41.18
 Giannis Antetokounmpo         Milwaukee Bucks  2021-22     39.34
           Paul George             LA Clippers  2021-22     39.34
         Kawhi Leonard             LA Clippers  2021-22     39.34
        Damian Lillard  Portland Trail Blazers  2021-22     39.34


Clean data, right salary ranges, all 5 seasons present. One thing to note — the $0.01M minimum salary is suspicious, that's likely a two-way contract player that slipped through. We'll catch it in the merge.

## Merge historical salary data to master

In [45]:
def merge_espn_salary(master_df, espn_df):
    """
    Join ESPN salary data onto our BBRef stats master table.
    
    We join on PLAYER_NAME + SEASON — both tables have one row
    per player per season so this is a clean one-to-one join.
    
    Left join means:
    - Every row in master is kept regardless of whether a salary match exists
    - Where a match IS found → salary columns get filled in
    - Where NO match is found → salary columns stay NaN
    
    This is safer than an inner join which would silently drop
    players with no salary match and shrink your dataset.
    """
    # Drop old salary columns if they exist from the previous attempt
    # so we don't end up with SALARY_M_x and SALARY_M_y duplicates
    cols_to_drop = ["SALARY", "SALARY_M", "SALARY_THIS_YEAR_M",
                    "SALARY_TOTAL_M", "YEARS_REMAINING", "SALARY_AVG_PER_YR_M"]
    master_df = master_df.drop(
        columns=[c for c in cols_to_drop if c in master_df.columns]
    ).copy()
    
    # Perform the join
    merged = master_df.merge(
        espn_df[["PLAYER_NAME", "SEASON", "SALARY", "SALARY_M"]],
        on  = ["PLAYER_NAME", "SEASON"],
        how = "left"
    )
    
    # ── Match rate report ────────────────────────────────────────────
    # This tells us how well our two data sources align on player names
    total     = len(merged)
    matched   = merged["SALARY_M"].notna().sum()
    unmatched = merged["SALARY_M"].isna().sum()
    pct       = matched / total * 100
    
    print(f"✓ Total rows:       {total}")
    print(f"✓ Salary matched:   {matched} ({pct:.1f}%)")
    print(f"✗ No salary found:  {unmatched} ({100-pct:.1f}%)")
    
    print(f"\nMatch rate by season:")
    for season, grp in merged.groupby("SEASON"):
        n_matched = grp["SALARY_M"].notna().sum()
        n_total   = len(grp)
        print(f"  {season}: {n_matched}/{n_total} ({n_matched/n_total*100:.0f}%)")
    
    return merged

master = merge_espn_salary(master, espn_clean)

✓ Total rows:       2688
✓ Salary matched:   2416 (89.9%)
✗ No salary found:  272 (10.1%)

Match rate by season:
  2021-22: 478/528 (91%)
  2022-23: 491/528 (93%)
  2023-24: 482/532 (91%)
  2024-25: 488/543 (90%)
  2025-26: 477/557 (86%)


In [46]:
# Surface unmatched players in 2025-26
# These are players in our BBRef stats who have no ESPN salary match
# Usually caused by slight name differences between the two sources
unmatched_current = master[
    (master["SALARY_M"].isna()) &
    (master["SEASON"] == "2025-26")
][["PLAYER_NAME", "TEAM", "BPM", "VORP"]].sort_values("BPM", ascending=False)

print(f"Unmatched in 2025-26: {len(unmatched_current)}")
print(unmatched_current.to_string(index=False))

Unmatched in 2025-26: 80
              PLAYER_NAME TEAM  BPM  VORP
          Robert Williams  POR  4.3   1.6
         Alperen AengA14n  HOU  4.1   3.6
       Michael Porter Jr.  BRK  3.0   2.1
           Gary Payton II  GSW  2.1   1.1
         Kevin Porter Jr.  MIL  2.0   1.3
           Moussa DiabatA  CHO  1.6   1.7
              Javon Small  MEM  1.3   0.7
              Nic Claxton  BRK  0.9   1.4
           Ron Harper Jr.  BOS  0.5   0.2
          Branden Carlson  OKC  0.4   0.3
        Jonas ValanAiAnas  DEN -0.4   0.3
         Tidjane SalaA14n  CHO -0.5   0.2
       Wendell Carter Jr.  ORL -0.5   0.9
        Derrick Jones Jr.  LAC -0.5   0.5
         Tristan Da Silva  ORL -0.6   0.7
        Jaren Jackson Jr.  2TM -0.6   0.5
      Kasparas JakuAionis  MIA -0.7   0.3
             DaRon Holmes  DEN -0.7   0.1
        Jaren Jackson Jr.  MEM -0.8   0.4
            GG Jackson II  MEM -1.2   0.2
              Dalen Terry  CHI -1.3   0.1
              Dalen Terry  2TM -1.7   0.0
         

89.9% match rate is solid — great improvement from 20%. Now let's fix the remaining issues. I can see three categories of problems in that list:
1. Still-broken unicode names — Alperen AengA14n, Dennis SchrAder, Moussa DiabatA etc. These slipped through our earlier fix because the broken strings are slightly different.
2. Name suffix mismatches — Michael Porter Jr., Gary Payton II, Jaren Jackson Jr. — ESPN has them one way, BBRef another.
3. Legitimate duplicates — Tony Bradley appearing 6 times, Rayan Rupert 4 times. These are the traded player rows we need to deduplicate.

In [47]:
# ── Fix 1: Additional unicode broken names ───────────────────────────────────
# These are new broken strings we didn't catch in notebook 01
# The pattern AengA14n = Şengün, SchrAder = Schröder etc.
additional_name_fixes = {
    # Broken unicode — these characters got mangled differently
    "Alperen AengA14n"        : "Alperen Sengun",
    "Dennis SchrAder"         : "Dennis Schroder",
    "Moussa DiabatA"          : "Moussa Diabate",
    "Jonas ValanAiAnas"       : "Jonas Valanciunas",
    "Tidjane SalaA14n"        : "Tidjane Salaun",
    "Kasparas JakuAionis"     : "Kasparas Jakucionis",
    "Hugo GonzAlez"           : "Hugo Gonzalez",
    "PacA me Dadiet"          : "Pacôme Dadiet",
    "Yanic Konan NiederhAuser": "Yanic Konan Niederhauser",
    "Nolan TraorA"            : "Nolan Traore",

    # Suffix mismatches — BBRef includes suffix, ESPN drops it (or vice versa)
    "Michael Porter Jr."      : "Michael Porter",
    "Gary Payton II"          : "Gary Payton",
    "Jaren Jackson Jr."       : "Jaren Jackson",
    "Wendell Carter Jr."      : "Wendell Carter",
    "Derrick Jones Jr."       : "Derrick Jones",
    "Larry Nance Jr."         : "Larry Nance",
    "Kevin Porter Jr."        : "Kevin Porter",
    "Ron Harper Jr."          : "Ron Harper",
    "Nick Smith Jr."          : "Nick Smith",
    "Kevin McCullar Jr."      : "Kevin McCullar",
    "GG Jackson II"           : "GG Jackson",
    "Robert Williams"         : "Robert Williams III",
    "Nic Claxton"             : "Nicolas Claxton",
    "Mo Wagner"               : "Moritz Wagner",
}

# Apply fixes to BOTH master and espn_clean so they match
# Teaching moment: we fix both sides of the join, not just one
# If we only fix master, the ESPN side still has the old name and won't match
master["PLAYER_NAME"]     = master["PLAYER_NAME"].replace(additional_name_fixes)
espn_clean["PLAYER_NAME"] = espn_clean["PLAYER_NAME"].replace(additional_name_fixes)

print("✓ Name fixes applied to both DataFrames")

# ── Fix 2: Deduplicate master again after name fixes ─────────────────────────
# Teaching moment: after fixing names, some rows that were different
# (e.g. "Jaren Jackson Jr." and "Jaren Jackson") are now identical
# We need to dedup again to remove those newly-created duplicates
before = len(master)
master = master.drop_duplicates(
    subset=["PLAYER_NAME", "SEASON", "TEAM"],
    keep="first"
).reset_index(drop=True)
print(f"✓ Removed {before - len(master)} duplicate rows after name fix")

✓ Name fixes applied to both DataFrames
✓ Removed 25 duplicate rows after name fix


In [48]:
# ── Re-run the merge with fixed names ────────────────────────────────────────
master = merge_espn_salary(master, espn_clean)

✓ Total rows:       2663
✓ Salary matched:   2481 (93.2%)
✗ No salary found:  182 (6.8%)

Match rate by season:
  2021-22: 491/528 (93%)
  2022-23: 503/528 (95%)
  2023-24: 494/532 (93%)
  2024-25: 509/543 (94%)
  2025-26: 484/532 (91%)


In [49]:
# ── Check remaining unmatched — should be much smaller now ───────────────────
unmatched_current = master[
    (master["SALARY_M"].isna()) &
    (master["SEASON"] == "2025-26")
][["PLAYER_NAME", "TEAM", "BPM", "VORP"]].sort_values("BPM", ascending=False)

print(f"Unmatched in 2025-26: {len(unmatched_current)}")
print(unmatched_current.head(20).to_string(index=False))

Unmatched in 2025-26: 48
      PLAYER_NAME TEAM  BPM  VORP
      Javon Small  MEM  1.3   0.7
       Ron Harper  BOS  0.5   0.2
  Branden Carlson  OKC  0.4   0.3
 Tristan Da Silva  ORL -0.6   0.7
     DaRon Holmes  DEN -0.7   0.1
       GG Jackson  MEM -1.2   0.2
      Dalen Terry  CHI -1.3   0.1
      Dalen Terry  2TM -1.7   0.0
       Jamal Cain  ORL -2.0   0.0
    Pacôme Dadiet  NYK -2.0   0.0
      Ron Holland  DET -2.0   0.0
 Chris Youngblood  2TM -2.2   0.0
       Pete Nance  MIL -2.3  -0.1
 Brooks Barnhizer  OKC -2.5   0.0
    Alijah Martin  TOR -2.5   0.0
    Aaron Holiday  HOU -2.6  -0.1
    Isaiah Livers  PHO -2.6   0.0
   Oscar Tshiebwe  UTA -2.7  -0.1
     Moussa Cisse  DAL -2.7  -0.1
 Chris Youngblood  OKC -2.8   0.0


93.2% match rate — that's excellent. Up from 89.9% to 93.2% after the name fixes.Looking at the remaining 48 unmatched, these fall into two categories:Not worth fixing — players with BPM below -1.5 are fringe/two-way contract players making minimum salary. They won't affect the CVI model meaningfully since we filter for quality players anyway.Worth fixing — the top few with decent BPM: Javon Small, Ron Harper, Branden Carlson, Tristan Da Silva, DaRon Holmes, GG Jackson, Dalen Terry, Pacome Dadiet.Let's fix those and then call this good enough:

In [50]:
# ── Final targeted name fixes for remaining unmatched players ────────────────
# Only fixing players with BPM > -1.5 since fringe players
# won't meaningfully affect the CVI model
final_name_fixes = {
    # These players exist in ESPN under slightly different names
    # Check ESPN salary page to find exact spelling used there
    "Javon Small"       : "Javon Small",      # may not be on ESPN (rookie/two-way)
    "Ron Harper"        : "Ron Harper Jr.",    # BBRef dropped Jr., ESPN keeps it
    "GG Jackson"        : "GG Jackson II",     # BBRef dropped II, ESPN keeps it
    "Dalen Terry"       : "Dalen Terry",       # check if ESPN has different spelling
    "Tristan Da Silva"  : "Tristan da Silva",  # lowercase 'd' in da
    "DaRon Holmes"      : "DaRon Holmes",      # capitalization difference
    "Pacôme Dadiet"     : "Pacome Dadiet",     # accent mark causing mismatch
}

# Apply only to master this time — we're matching master names TO espn names
# So we need master to use whatever spelling ESPN has
master["PLAYER_NAME"] = master["PLAYER_NAME"].replace(final_name_fixes)

# Re-run merge one final time
master = merge_espn_salary(master, espn_clean)

# Final match rate check
total     = len(master)
matched   = master["SALARY_M"].notna().sum()
print(f"✓ Final match rate: {matched}/{total} ({matched/total*100:.1f}%)")

print(f"\nStill unmatched in 2025-26 (below threshold — acceptable):")
print(master[
    (master["SALARY_M"].isna()) &
    (master["SEASON"] == "2025-26")
][["PLAYER_NAME", "TEAM", "BPM"]].sort_values("BPM", ascending=False).to_string(index=False))

✓ Total rows:       2663
✓ Salary matched:   2484 (93.3%)
✗ No salary found:  179 (6.7%)

Match rate by season:
  2021-22: 491/528 (93%)
  2022-23: 503/528 (95%)
  2023-24: 494/532 (93%)
  2024-25: 510/543 (94%)
  2025-26: 486/532 (91%)
✓ Final match rate: 2484/2663 (93.3%)

Still unmatched in 2025-26 (below threshold — acceptable):
            PLAYER_NAME TEAM  BPM
            Javon Small  MEM  1.3
         Ron Harper Jr.  BOS  0.5
        Branden Carlson  OKC  0.4
           DaRon Holmes  DEN -0.7
          GG Jackson II  MEM -1.2
            Dalen Terry  CHI -1.3
            Dalen Terry  2TM -1.7
            Ron Holland  DET -2.0
             Jamal Cain  ORL -2.0
       Chris Youngblood  2TM -2.2
             Pete Nance  MIL -2.3
          Alijah Martin  TOR -2.5
       Brooks Barnhizer  OKC -2.5
          Aaron Holiday  HOU -2.6
          Isaiah Livers  PHO -2.6
         Oscar Tshiebwe  UTA -2.7
           Moussa Cisse  DAL -2.7
       Chris Youngblood  OKC -2.8
           Kobe San

In [51]:
# ── Add multi-year contract metrics from BBRef contracts page ─────────────────
# ESPN gives us salary per season which is great for historical matching
# BBRef contracts page gives us total contract value and years remaining
# We need both — join BBRef contract summary onto current season rows only

# Get contract summary from the wide contracts table we scraped earlier
contract_summary = contracts[[
    "PLAYER_NAME", "TEAM",
    "SALARY_THIS_YEAR_M",
    "SALARY_TOTAL_M",
    "YEARS_REMAINING",
    "SALARY_AVG_PER_YR_M"
]].copy()

# Fix names in contract summary to match our standard
contract_summary["PLAYER_NAME"] = (contract_summary["PLAYER_NAME"]
                                   .replace(additional_name_fixes)
                                   .replace(final_name_fixes))

# Join onto master — only current season rows will match
# since BBRef contracts page only has current + future years
master = master.merge(
    contract_summary,
    on  = ["PLAYER_NAME", "TEAM"],
    how = "left"
)

print(f"✓ Contract metrics added")
print(f"\nSample — current season with full contract details:")
print(master[master["SEASON"] == "2025-26"][[
    "PLAYER_NAME", "TEAM", "SALARY_M",
    "SALARY_TOTAL_M", "YEARS_REMAINING", "SALARY_AVG_PER_YR_M"
]].sort_values("SALARY_M", ascending=False).head(10).to_string(index=False))

✓ Contract metrics added

Sample — current season with full contract details:
           PLAYER_NAME TEAM  SALARY_M  SALARY_TOTAL_M  YEARS_REMAINING  SALARY_AVG_PER_YR_M
         Stephen Curry  GSW     59.61          122.19              2.0                61.10
          Nikola Jokic  DEN     55.22          177.10              3.0                59.03
           Joel Embiid  PHI     55.22          243.47              4.0                60.87
          Kevin Durant  HOU     54.71          144.71              3.0                48.24
           Luka Doncic  LAL     54.13          207.35              4.0                51.84
    Karl-Anthony Towns  NYK     54.13          171.24              3.0                57.08
          Jimmy Butler  GSW     54.13          110.96              2.0                55.48
         Anthony Davis  DAL     54.13             NaN              NaN                  NaN
 Giannis Antetokounmpo  MIL     54.13          175.37              3.0                58.46
  

## Save

In [52]:
# ── Save final processed dataset ─────────────────────────────────────────────
# This goes to data/processed/ — our cleaned, merged, ready-to-use dataset
# Everything from here forward reads from this file
os.makedirs("../data/processed", exist_ok=True)
master.to_csv("../data/processed/master_with_salary.csv", index=False)

print(f"✓ Saved {len(master)} rows → data/processed/master_with_salary.csv")
print(f"\nFinal column list ({master.shape[1]} columns):")
print(master.columns.tolist())
print(f"\nSeason breakdown:")
print(master.groupby(["SEASON", "DATA_TYPE"])["PLAYER_NAME"].count().to_string())

✓ Saved 2668 rows → data/processed/master_with_salary.csv

Final column list (61 columns):
['PLAYER_NAME', 'AGE', 'TEAM', 'POSITION', 'G', 'GAMES_STARTED', 'MIN', 'FG', 'FGA', 'FG_PCT', '3P', '3PA', '3P_PCT', '2P', '2PA', '2P_PCT', 'EFG_PCT', 'FT', 'FTA', 'FT_PCT', 'ORB', 'DRB', 'REB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PTS', 'AWARDS', 'SEASON', 'PER', 'TS_PCT', '3PAR', 'FTR', 'ORB_PCT', 'DRB_PCT', 'TRB_PCT', 'AST_PCT', 'STL_PCT', 'BLK_PCT', 'TOV_PCT', 'USG_PCT', 'OWS', 'DWS', 'WS', 'WS_48', 'OBPM', 'DBPM', 'BPM', 'VORP', 'LAST_TEAM', 'DATA_TYPE', 'POSITION_GROUP', 'PLAYER_NAME_RAW', 'SALARY', 'SALARY_M', 'SALARY_THIS_YEAR_M', 'SALARY_TOTAL_M', 'YEARS_REMAINING', 'SALARY_AVG_PER_YR_M']

Season breakdown:
SEASON   DATA_TYPE
2021-22  training     529
2022-23  training     528
2023-24  training     532
2024-25  current      543
2025-26  current      536


In [54]:
# Fix DATA_TYPE — 2024-25 is a completed season so it's training data
# Only 2025-26 (the active season) is current
master["DATA_TYPE"] = master["SEASON"].apply(
    lambda s: "current" if s == "2025-26" else "training"
)

print("✓ DATA_TYPE fixed")
print(master.groupby(["SEASON", "DATA_TYPE"])["PLAYER_NAME"].count().to_string())

# Resave with the fix
master.to_csv("../data/processed/master_with_salary.csv", index=False)
print(f"\n✓ Resaved — {len(master)} rows")

✓ DATA_TYPE fixed
SEASON   DATA_TYPE
2021-22  training     529
2022-23  training     528
2023-24  training     532
2024-25  training     543
2025-26  current      536

✓ Resaved — 2668 rows


In [55]:
# Save everything to CSV so notebook 03 never needs to re-scrape
os.makedirs("../data/raw", exist_ok=True)

espn_clean.to_csv("../data/raw/espn_salaries.csv", index=False)
contracts_wide.to_csv("../data/raw/bbref_contracts_wide.csv", index=False)
contracts_long.to_csv("../data/raw/bbref_contracts_long.csv", index=False)

print(f"✓ Saved espn_salaries.csv       — {len(espn_clean)} rows")
print(f"✓ Saved bbref_contracts_wide.csv — {len(contracts_wide)} rows")
print(f"✓ Saved bbref_contracts_long.csv — {len(contracts_long)} rows")

✓ Saved espn_salaries.csv       — 2466 rows
✓ Saved bbref_contracts_wide.csv — 527 rows
✓ Saved bbref_contracts_long.csv — 1242 rows


## Final Thoughts

#### Why is 93% good enough?

In data science, 100% data coverage is almost never achievable when joining across different sources. The 7% unmatched players are almost entirely fringe players on minimum contracts (around $1-2M). Since our CVI model is designed to evaluate meaningful contracts, these players would likely be filtered out anyway when we apply the 20+ games and 15+ minutes filters. Spending hours chasing the last 7% would be diminishing returns — better to move forward and note the coverage gap in your model documentation.

#### Expiring Contracts vs. One year Deal - Does it matter? 

Yes it absolutely matters for CVI. An expiring contract is fundamentally different from a multi-year deal even if the annual salary is the same. Here's why:
For the team, an expiring contract actually has hidden value — it creates cap space the following summer, can be used as a trade piece (expiring deals are attractive to teams wanting to dump salary), and carries zero future risk. A player on an expiring deal could be terrible and the team just moves on. Whereas a bad player on a 4-year deal is a franchise anchor around their neck.
So for the CVI model, YEARS_REMAINING = 1 should actually get a slight boost on the apron impact score because the financial risk is contained to one year. The model currently penalizes Anthony Davis for missing SALARY_TOTAL_M data but it should recognize him as a low roster-flexibility-risk contract despite his injury history.
We'll handle this in feature engineering by treating YEARS_REMAINING = 1 as a special case.

On contract type (player option, team option, etc.):
100% a v2 feature and a smart one. Here's how each type would affect CVI differently:
Fully guaranteed:Neutral baseline
Player option: Slight negative for team — player holds the leverage
Team option: Slight positive for team — they control the decision
Two-way: Not scored — below model threshold
Qualifying offer: Neutral — player likely to leave
This data exists on BBRef and Spotrac. We'll flag it for v2.